# How to Validate Before Encoding

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/how-to/validate_before_encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Fhow-to%2Fvalidate_before_encoding.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/how-to/validate_before_encoding.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>


Three complementary tools:
- `preprocessing_report()` — structured dataset audit
- `check_compatibility()` — encoder-specific range checks
- `DataSchema` — define, infer, validate, and serialise schemas

In [ ]:
!pip install -q quprep

In [ ]:
import warnings

import numpy as np

import quprep as qd
from quprep import QuPrepWarning

print(f"quprep {qd.__version__}")

rng = np.random.default_rng(0)
X = rng.uniform(0, 10, (60, 6))   # values outside [0, π] — angle encoder will flag this
X[0, 2] = np.nan
y = np.array([0] * 54 + [1] * 6)  # 9:1 imbalance

ds = qd.NumpyIngester().load(X, y=y)

## 1. `preprocessing_report()` — dataset audit

In [ ]:
report = qd.preprocessing_report(ds, encoder=qd.AngleEncoder(), qubit_budget=4)
print(f"Issues : {report.n_issues}")
for rec in report.recommendations:
    print(f"  → {rec}")

## 2. `check_compatibility()` — encoder range check

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    ds_clean = qd.Pipeline(
        cleaner=qd.Imputer(strategy="mean"),
        normalizer=qd.Scaler(strategy="minmax_pi"),
    ).fit_transform(ds).dataset

compat = qd.check_compatibility(qd.AngleEncoder(), ds_clean)
print(f"Compatible : {compat.is_compatible}")
print(f"Warnings   : {len(compat.warnings)}")
print(f"Errors     : {len(compat.errors)}")

## 3. `verify_encoding()` — post-encoding invariant check

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    result = qd.Pipeline(
        cleaner=qd.Imputer(strategy="mean"),
        normalizer=qd.Scaler(strategy="minmax_pi"),
        encoder=qd.AngleEncoder(),
    ).fit_transform(ds)

verify = qd.verify_encoding(result.encoded, qd.AngleEncoder())
print(f"Passed : {verify.passed}")
for check in verify.checks:
    status = "✓" if check["passed"] else "✗"
    print(f"  {status} {check['name']}: {check['detail']}")

## 4. `DataSchema` — infer, save, and validate

In [ ]:
schema = qd.DataSchema.infer(ds_clean)
print(f"Columns             : {len(schema.features)}")
print(f"First feature spec  : {schema.to_dict()[0]}")

schema_json = schema.to_json()
schema2 = qd.DataSchema.from_json(schema_json)

print("\nValidating against schema:")
try:
    schema2.validate(ds_clean)
    print("  Valid  : True")
    print("  Errors : 0")
except Exception as e:
    print(f"  Invalid: {e}")